In [1]:
!pip install -q langchain langgraph langchain-groq wikipedia requests typing_extensions

zsh:1: command not found: pip


In [2]:
import os
import wikipedia
import requests
import datetime

from typing import Annotated
from typing_extensions import TypedDict

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

In [ ]:
import os
from dotenv import load_dotenv

# Fix SSL certificate error
if "SSL_CERT_FILE" in os.environ:
    del os.environ["SSL_CERT_FILE"]
if "SSL_CERT_DIR" in os.environ:
    del os.environ["SSL_CERT_DIR"]

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
@tool
def wikipedia_agent(query: str) -> str:
    """Search Wikipedia for factual knowledge."""
    try:
        return wikipedia.summary(query, sentences=3)
    except:
        return "Information not found."

In [5]:
@tool
def weather_agent(city: str) -> str:
    """Get weather forecast from an online API."""
    
    geo = requests.get(
        f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
    ).json()

    if "results" not in geo:
        return "City not found."

    lat = geo["results"][0]["latitude"]
    lon = geo["results"][0]["longitude"]

    weather = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    ).json()

    temp = weather["current_weather"]["temperature"]

    return f"Current temperature in {city} is {temp}°C."


In [6]:
@tool
def math_agent(expression: str) -> str:
    """Solve math expressions."""
    try:
        return str(eval(expression))
    except:
        return "Invalid math expression."


In [7]:
@tool
def time_agent(city: str) -> str:
    """Get current time."""
    now = datetime.datetime.now()
    return f"Current system time is {now.strftime('%H:%M:%S')}."

In [8]:
@tool
def news_agent(topic: str) -> str:
    """Get latest news headlines."""
    
    url = f"https://newsapi.org/v2/everything?q={topic}&apiKey=YOUR_NEWS_API"
    
    try:
        data = requests.get(url).json()
        articles = data["articles"][:3]

        headlines = [a["title"] for a in articles]

        return "\n".join(headlines)

    except:
        return "Could not fetch news."

In [9]:
@tool
def coding_agent(task: str) -> str:
    """Provide programming help."""
    return f"Programming request detected: {task}"


tools = [
    wikipedia_agent,
    weather_agent,
    math_agent,
    time_agent,
    news_agent,
    coding_agent
]


In [10]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [11]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

llm_with_tools = llm.bind_tools(tools)

In [12]:
system_prompt = SystemMessage(
content="""
You are an AI with specialized agents.

Use tools when appropriate:

wikipedia_agent → facts, people, history, science
weather_agent → weather or forecasts
math_agent → calculations
time_agent → time queries
news_agent → latest news
coding_agent → programming help

If the user just chats, respond normally.
"""
)


In [13]:
def chatbot(state: State):
    messages = [system_prompt] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


In [14]:
def should_continue(state: State):
    last = state["messages"][-1]

    if getattr(last, "tool_calls", None):
        return "tools"

    return END

In [15]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges(
    "chatbot",
    should_continue,
    {"tools": "tools", END: END},
)
graph_builder.add_edge("tools", "chatbot")
graph = graph_builder.compile()


In [16]:
def chat(user_input, history):
    history.append(HumanMessage(content=user_input))
    result = graph.invoke({"messages": history})
    history = result["messages"]
    print("\nAssistant:\n")
    print(history[-1].content)
    print()
    return history
print("\n Multi-Scenario Agentic AI Ready!\n")
history = []
while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit"]:
        break
    history = chat(user_input, history)


 Multi-Scenario Agentic AI Ready!


Assistant:

How can I assist you today? Do you have any questions or need help with something specific?


Assistant:

It's nice to meet you. Is there something I can help you with or would you like to chat?


Assistant:

I hope this information helps! Let me know if you have any other questions.


Assistant:

Kuala Lumpur is the cultural, financial and economic centre of Malaysia and one of the fastest-growing metropolitan regions in Southeast Asia, both in terms of economic development and population growth. It is home to the Petronas Twin Towers, the tallest twin towers in the world, as well as many other notable landmarks and attractions. Let me know if you have any other questions!


Assistant:

The Kolkata Knight Riders have won the IPL title twice, in 2012 and 2014. They have a strong fan base and are known for their exciting brand of cricket. Let me know if you have any other questions!


Assistant:

The Kolkata Knight Riders (KKR) are a prof